# 23 — Independent motion review and clean clip package

This CPU notebook completes Stage 2 of the post-V6-C roadmap. It downloads the
verified Notebook 22 report from pinned GitHub commit `5146096`, checks its
SHA-256, repeats the pose audit, and creates a focused independent review set:

- every one of the 20 supported intervals;
- 12 deterministic `unknown` intervals, more than 10% of the 98 unknown rows.

Use a **CPU runtime**. If Notebook 22's videos are still present under
`/content/calorie_youtube_long_v2_work/videos`, they are reused. Otherwise the
notebook downloads 360p copies only for human motion/boundary review. The
training arrays always come from Notebook 22's already extracted 720p pose
landmarks.

For every review clip, inspect the motion, confirm/change/reject the canonical
label, and adjust conservative edge trims if setup or transition motion leaks
inside. Export is blocked until every target has a saved decision. The output
is `calorie_youtube_long_workouts_v2_reviewed_clips.zip`.


In [ ]:
%pip install -q --upgrade yt-dlp pandas==2.2.3 numpy ipywidgets


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import Video, clear_output, display
from google.colab import files

RUNTIME = 'CPU'
PINNED_REPORT_COMMIT = '5146096fed1282fc3b82a8878687613cf3ccfe33'
REPORT_SHA256 = 'f97bf6844f465e7c4bc09f3131bd41edbcda32e20f334ae59a2dfc95b5a48506'
REPORT_NAME = 'calorie_youtube_long_workouts_v2_review_landmarks.zip'
REPORT_URL = (
    'https://raw.githubusercontent.com/bariscelikk1/cALorie/'
    f'{PINNED_REPORT_COMMIT}/ml/artifacts/youtube_long_v2/landmarks/{REPORT_NAME}'
)
WORK_ROOT = Path('/content/calorie_youtube_long_v2_review')
REPORT_PATH = WORK_ROOT / REPORT_NAME
EXTRACT_ROOT = WORK_ROOT / 'notebook22_report'
VIDEO_ROOT = Path('/content/calorie_youtube_long_v2_work/videos')
REVIEW_CLIP_ROOT = WORK_ROOT / 'review_clips'
OUTPUT_ROOT = Path('/content/calorie_youtube_long_workouts_v2_reviewed_clips')
for path in (WORK_ROOT, EXTRACT_ROOT, VIDEO_ROOT, REVIEW_CLIP_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CLASSES = [
    'squat', 'push_up', 'jumping_jack', 'pull_up', 'lunge', 'sit_up',
    'shoulder_press', 'dumbbell_row', 'tricep_extension', 'bicep_curl',
    'lateral_raise', 'bench_press', 'deadlift', 'lat_pulldown', 'unknown',
]
SOURCES = {
    'zxqCV_eQHZY': 'legally_swole__zxqCV_eQHZY',
    'Fk1wnNX4wQ8': 'daniel_nikolic__Fk1wnNX4wQ8',
}
print('Runtime:', RUNTIME)
print('Pinned Notebook 22 report commit:', PINNED_REPORT_COMMIT)
print('Output:', OUTPUT_ROOT.with_suffix('.zip'))


## Download and verify the Notebook 22 report

The archive is reused when its checksum already matches. Both the outer ZIP
hash and every hash stored inside its `SHA256SUMS.txt` must pass.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


if not REPORT_PATH.exists() or sha256(REPORT_PATH) != REPORT_SHA256:
    temporary = REPORT_PATH.with_suffix('.zip.part')
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(REPORT_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=300) as response, temporary.open('wb') as target:
        shutil.copyfileobj(response, target)
    assert sha256(temporary) == REPORT_SHA256, 'Notebook 22 report checksum mismatch'
    temporary.replace(REPORT_PATH)

assert sha256(REPORT_PATH) == REPORT_SHA256
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(REPORT_PATH) as archive:
    bad = archive.testzip()
    assert bad is None, f'Corrupt ZIP member: {bad}'
    archive.extractall(EXTRACT_ROOT)

for line in (EXTRACT_ROOT / 'SHA256SUMS.txt').read_text().splitlines():
    expected, relative = line.split(maxsplit=1)
    relative = relative.lstrip('*')
    assert sha256(EXTRACT_ROOT / relative) == expected, f'Internal checksum mismatch: {relative}'
print('Verified:', REPORT_PATH.name, REPORT_SHA256)


## Repeat the pose audit and choose independent-review targets

The review set contains every supported interval and a stable hash-ranked
sample of 12 sufficiently long unknown intervals. Pose failures are measured
again from the raw `detected` mask rather than trusting the manifest summary.


In [ ]:
def max_false_run(mask):
    best = current = 0
    for value in mask:
        if value:
            current = 0
        else:
            current += 1
            best = max(best, current)
    return best


rows = []
pose_rows = []
for label_path in sorted((EXTRACT_ROOT / 'first_pass_labels').glob('*.csv')):
    stem = label_path.name.removesuffix('.labels.csv')
    frame = pd.read_csv(label_path)
    archive = np.load(EXTRACT_ROOT / 'continuous_landmarks' / f'{stem}.npz', allow_pickle=False)
    timestamps = archive['timestamps_s']
    detected = archive['detected']
    visibility = archive['landmarks'][:, :, 3]
    target_fps = float(archive['target_fps'])
    assert np.isfinite(archive['landmarks']).all()
    for index, row in frame.iterrows():
        mask = (timestamps >= float(row.start_s)) & (timestamps < float(row.end_s))
        selected = detected[mask]
        pose_rows.append({
            'review_id': f'{row.video_id}:{index:03d}', 'video_id': row.video_id,
            'row_index': index, 'detection_rate': float(selected.mean()),
            'max_missing_s': max_false_run(selected) / target_fps,
            'median_visible_joints': float(np.median((visibility[mask] >= 0.5).sum(axis=1))),
        })
        item = row.to_dict()
        item.update({'row_index': index, 'stem': stem, 'review_id': f'{row.video_id}:{index:03d}'})
        rows.append(item)

timeline = pd.DataFrame(rows)
pose_audit = pd.DataFrame(pose_rows)
timeline = timeline.merge(pose_audit, on=['review_id', 'video_id', 'row_index'])
timeline['duration_s'] = timeline.end_s - timeline.start_s

supported = timeline[timeline.current_tcn_label != 'unknown'].copy()
unknown_pool = timeline[
    (timeline.current_tcn_label == 'unknown') & (timeline.duration_s >= 12)
].copy()
unknown_pool['sample_key'] = unknown_pool.review_id.map(
    lambda value: hashlib.sha256(f'notebook23:{value}'.encode()).hexdigest()
)
unknown_sample = unknown_pool.sort_values('sample_key').head(12)
review_targets = pd.concat([supported, unknown_sample], ignore_index=True).drop_duplicates('review_id')
review_targets = review_targets.sort_values(['video_id', 'start_s']).reset_index(drop=True)
review_targets.to_csv(WORK_ROOT / 'review_targets.csv', index=False)

print('All timeline rows:', len(timeline))
print('Supported review targets:', len(supported))
print('Unknown review targets:', int((review_targets.current_tcn_label == 'unknown').sum()))
display(review_targets[[
    'review_id', 'start_time', 'end_time', 'fine_label', 'current_tcn_label',
    'detection_rate', 'median_visible_joints'
]])
assert len(supported) == 20
assert (review_targets.current_tcn_label == 'unknown').sum() == 12
assert supported.detection_rate.min() > 0.99


## Reuse or download review video, then build small review clips

Existing Notebook 22 videos are preferred. A missing source is downloaded at
up to 360p because this copy is only for human motion and boundary review; it
never replaces the 720p-derived landmark arrays.


In [ ]:
VIDEO_SUFFIXES = {'.mp4', '.mkv', '.webm', '.mov'}


def find_video(stem):
    matches = [
        path for path in VIDEO_ROOT.glob(f'{stem}.*')
        if path.suffix.lower() in VIDEO_SUFFIXES and path.stat().st_size > 5_000_000
    ]
    assert len(matches) <= 1, (stem, matches)
    return matches[0] if matches else None


def download_review_video(video_id, stem):
    existing = find_video(stem)
    if existing:
        print('Reusing:', existing.name)
        return existing
    template = VIDEO_ROOT / f'{stem}.%(ext)s'
    url = f'https://www.youtube.com/watch?v={video_id}'
    attempts = [
        ([], 'best[height<=360][ext=mp4]/best[height<=360]'),
        (['--extractor-args', 'youtube:player_client=android_vr'], 'best[height<=360]/worst'),
    ]
    failures = []
    for extra, format_rule in attempts:
        command = [
            'yt-dlp', '--no-playlist', '--no-progress', '--retries', '10',
            '--fragment-retries', '10', '--retry-sleep', 'fragment:exp=1:8',
            '-f', format_rule, '--merge-output-format', 'mp4', '-o', str(template),
            *extra, url,
        ]
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        existing = find_video(stem)
        if result.returncode == 0 and existing:
            return existing
        failures.append('\n'.join(result.stdout.splitlines()[-30:]))
    raise RuntimeError(f'Could not download review video {video_id}:\n' + '\n\n'.join(failures))


video_paths = {video_id: download_review_video(video_id, stem) for video_id, stem in SOURCES.items()}
for row in review_targets.itertuples(index=False):
    clip_path = REVIEW_CLIP_ROOT / f"{row.review_id.replace(':', '__')}.mp4"
    if clip_path.exists() and clip_path.stat().st_size > 10_000:
        continue
    context_start = max(0.0, float(row.start_s) - 2.0)
    context_duration = float(row.end_s) - context_start + 2.0
    result = subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-ss', f'{context_start:.3f}', '-i', str(video_paths[row.video_id]),
        '-t', f'{context_duration:.3f}', '-vf', 'scale=-2:360', '-an',
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '28', str(clip_path),
    ], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert result.returncode == 0 and clip_path.exists(), result.stdout
print('Review clips ready:', len(list(REVIEW_CLIP_ROOT.glob('*.mp4'))))


## Save every independent review decision

Choose a target, watch its full-motion clip, enter a stable reviewer ID, and
save. `accept` retains the proposed label; `change` uses the selected label;
`reject` excludes the interval. Edge trims remove setup/transition frames from
the training clip without pretending that the coarse source timeline has
frame-perfect boundaries.


In [ ]:
PROGRESS_PATH = WORK_ROOT / 'review_progress.json'
decisions = json.loads(PROGRESS_PATH.read_text()) if PROGRESS_PATH.exists() else {}
target_by_id = review_targets.set_index('review_id')

reviewer = widgets.Text(description='Reviewer:', placeholder='name_or_id')
selector = widgets.Dropdown(
    options=[(f"{row.review_id} | {row.current_tcn_label} | {row.fine_label}", row.review_id)
             for row in review_targets.itertuples(index=False)],
    description='Target:', layout=widgets.Layout(width='95%'),
)
decision = widgets.Dropdown(options=['accept', 'change', 'reject'], description='Decision:')
second_label = widgets.Dropdown(options=CLASSES, description='Label:')
confidence = widgets.Dropdown(options=['high', 'medium', 'low'], description='Confidence:')
trim_start = widgets.BoundedFloatText(value=2.0, min=0.0, max=10.0, step=0.5, description='Trim start:')
trim_end = widgets.BoundedFloatText(value=2.0, min=0.0, max=10.0, step=0.5, description='Trim end:')
notes = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='95%'))
save_button = widgets.Button(description='Save and next', button_style='success')
status = widgets.HTML()
viewer = widgets.Output()


def load_target(change=None):
    review_id = selector.value
    row = target_by_id.loc[review_id]
    prior = decisions.get(review_id, {})
    decision.value = prior.get('decision', 'accept')
    second_label.value = prior.get('second_label', row.current_tcn_label)
    confidence.value = prior.get('confidence', row.annotation_confidence)
    default_trim = 2.0 if row.current_tcn_label != 'unknown' else 4.0
    trim_start.value = float(prior.get('trim_start_s', default_trim))
    trim_end.value = float(prior.get('trim_end_s', default_trim))
    notes.value = prior.get('notes', '')
    clip_path = REVIEW_CLIP_ROOT / f"{review_id.replace(':', '__')}.mp4"
    with viewer:
        clear_output(wait=True)
        print(f"{row.start_time}-{row.end_time} | proposed={row.current_tcn_label} | fine={row.fine_label}")
        print(f"pose detection={row.detection_rate:.3%} | median visible joints={row.median_visible_joints:.0f}/17")
        display(Video(str(clip_path), embed=True, width=720))
    status.value = f'<b>Saved {len(decisions)} / {len(review_targets)}</b>'


def save_current(_):
    if not reviewer.value.strip():
        status.value = '<b style="color:red">Enter a reviewer ID first.</b>'
        return
    review_id = selector.value
    row = target_by_id.loc[review_id]
    remaining = float(row.duration_s) - trim_start.value - trim_end.value
    if decision.value != 'reject' and remaining < 4.27:
        status.value = '<b style="color:red">Trimmed clip must remain at least 4.27 seconds.</b>'
        return
    decisions[review_id] = {
        'review_id': review_id, 'reviewer': reviewer.value.strip(),
        'decision': decision.value, 'second_label': second_label.value,
        'confidence': confidence.value, 'trim_start_s': trim_start.value,
        'trim_end_s': trim_end.value, 'notes': notes.value.strip(),
    }
    PROGRESS_PATH.write_text(json.dumps(decisions, indent=2, sort_keys=True) + '\n')
    ids = list(target_by_id.index)
    current = ids.index(review_id)
    status.value = f'<b>Saved {len(decisions)} / {len(review_targets)}</b>'
    if current + 1 < len(ids):
        selector.value = ids[current + 1]


selector.observe(load_target, names='value')
save_button.on_click(save_current)
display(widgets.VBox([
    reviewer, selector, viewer,
    widgets.HBox([decision, second_label, confidence]),
    widgets.HBox([trim_start, trim_end]), notes, save_button, status,
]))
load_target()


## Export source-group-safe training clips

Run this only after the review widget reports every target saved. Rejected rows
are omitted. Accepted clips must retain at least 64 frames; supported clips
also require at least 95% direct pose detections. The package keeps both source
videos as distinct `group_id` values for split-before-windowing.


In [ ]:
assert PROGRESS_PATH.exists(), 'Complete the review widget first.'
decisions = json.loads(PROGRESS_PATH.read_text())
missing = sorted(set(review_targets.review_id) - set(decisions))
assert not missing, f'Unsaved review targets ({len(missing)}): {missing[:8]}'

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
(OUTPUT_ROOT / 'clips').mkdir()

manifest_rows = []
review_rows = []
for row in review_targets.itertuples(index=False):
    item = decisions[row.review_id]
    review_rows.append({**row._asdict(), **item})
    if item['decision'] == 'reject':
        continue
    canonical = row.current_tcn_label if item['decision'] == 'accept' else item['second_label']
    start_s = float(row.start_s) + float(item['trim_start_s'])
    end_s = float(row.end_s) - float(item['trim_end_s'])
    archive = np.load(
        EXTRACT_ROOT / 'continuous_landmarks' / f'{row.stem}.npz', allow_pickle=False
    )
    timestamps = archive['timestamps_s']
    selected = np.flatnonzero((timestamps >= start_s) & (timestamps < end_s))
    assert len(selected) >= 64, (row.review_id, len(selected))
    direct_rate = float(archive['detected'][selected].mean())
    if canonical != 'unknown':
        assert direct_rate >= 0.95, (row.review_id, direct_rate)
    clip_id = f"ytlongv2_{row.video_id}_{row.row_index:03d}_{canonical}"
    class_root = OUTPUT_ROOT / 'clips' / canonical
    class_root.mkdir(parents=True, exist_ok=True)
    output_path = class_root / f'{clip_id}.npz'
    np.savez_compressed(
        output_path,
        landmarks=archive['landmarks'][selected].astype(np.float32),
        timestamps_s=(timestamps[selected] - start_s).astype(np.float32),
        detected=archive['detected'][selected],
    )
    manifest_rows.append({
        'clip_id': clip_id,
        'source_dataset': 'youtube_long_workouts_v2_reviewed',
        'source_class': row.fine_label,
        'canonical_class': canonical,
        'original_path': f"youtube:{row.video_id}#{start_s:.3f}-{end_s:.3f}",
        'output_path': str(output_path.relative_to(OUTPUT_ROOT)),
        'fingerprint': row.video_id,
        'status': 'ok',
        'source_fps': float(archive['target_fps']),
        'sampled_frames': len(selected),
        'detection_rate': direct_rate,
        'duration_s': end_s - start_s,
        'group_id': f'youtube:{row.video_id}',
        'reviewer': item['reviewer'],
        'review_decision': item['decision'],
        'review_confidence': item['confidence'],
    })

manifest = pd.DataFrame(manifest_rows)
assert manifest.group_id.nunique() == 2
assert manifest.output_path.map(lambda value: (OUTPUT_ROOT / value).exists()).all()
manifest.to_csv(OUTPUT_ROOT / 'manifest.csv', index=False)
pd.DataFrame(review_rows).to_csv(OUTPUT_ROOT / 'independent_review_decisions.csv', index=False)
pose_audit.to_csv(OUTPUT_ROOT / 'pose_interval_audit.csv', index=False)
metadata = {
    'notebook': '23_colab_review_build_youtube_long_v2.ipynb',
    'runtime': 'CPU',
    'source_report_sha256': REPORT_SHA256,
    'source_report_commit': PINNED_REPORT_COMMIT,
    'review_targets': len(review_targets),
    'exported_clips': len(manifest),
    'groups': sorted(manifest.group_id.unique()),
    'training_allowed': True,
    'split_rule': 'keep each group_id wholly within one train/validation/test partition',
}
(OUTPUT_ROOT / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')

checksum_lines = []
for path in sorted(p for p in OUTPUT_ROOT.rglob('*') if p.is_file()):
    checksum_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}')
(OUTPUT_ROOT / 'SHA256SUMS.txt').write_text('\n'.join(checksum_lines) + '\n')

display(manifest.groupby('canonical_class').agg(
    clips=('clip_id', 'count'), frames=('sampled_frames', 'sum'),
    detection_min=('detection_rate', 'min'), detection_mean=('detection_rate', 'mean'),
).round(4))

archive_path = shutil.make_archive(str(OUTPUT_ROOT), 'zip', OUTPUT_ROOT)
print('Created:', archive_path, f'({Path(archive_path).stat().st_size / 1e6:.1f} MB)')
files.download(archive_path)
